# Lesson 10: Model Deployment - Comparing Models for Production

This notebook covers:
- Training multiple models (EfficientNet-B2 vs ViT-B/16)
- Comparing model performance
- Measuring inference speed
- Making deployment decisions
- Production considerations

## What is Model Deployment?

**Deployment** = Making your model available for real-world use

### Deployment Considerations:
1. **Accuracy**: Does it work well enough?
2. **Speed**: Fast enough for real-time use?
3. **Size**: Small enough to deploy?
4. **Resources**: GPU needed or CPU sufficient?
5. **Cost**: Compute costs acceptable?

## Models We'll Compare:

### EfficientNet-B2
- **Type**: Convolutional Neural Network (CNN)
- **Parameters**: ~9M
- **Strengths**: Fast, efficient, proven
- **Use case**: Mobile/edge deployment

### Vision Transformer (ViT-B/16)
- **Type**: Transformer (attention-based)
- **Parameters**: ~86M
- **Strengths**: State-of-the-art accuracy
- **Use case**: Cloud/server deployment

## Goal:
Find the **best model** for deploying a food classifier based on:
- Test accuracy
- Inference time
- Model size
- Resource requirements

## Setup and Imports

In [ ]:
# Check Python version
import sys
sys.version

In [ ]:
# Check PyTorch and CUDA
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# Version requirements
try:
    import torch
    import torchvision
    assert int(torch.__version__.split(".")[1]) >= 12, "torch version should be 1.12+"
    assert int(torchvision.__version__.split(".")[1]) >= 13, "torchvision version should be 0.13+"
except AssertionError as e:
    print(f"[WARNING] {e}")

In [ ]:
# Continue with imports
import torch
import torchvision
from torch import nn
from torchvision import transforms
import matplotlib.pyplot as plt

# Try to get torchinfo
try:
    from torchinfo import summary
except:
    print("[INFO] Couldn't find torchinfo... installing it.")
    !pip install -q torchinfo
    from torchinfo import summary

# Import going_modular utilities
try:
    from going_modular import data_setup, engine
    from going_modular.utils import download_data, set_seeds
    from helper_functions import plot_loss_curves
except:
    print("[INFO] Couldn't find going_modular... downloading from GitHub.")
    !git clone https://github.com/mrdbourke/pytorch-deep-learning
    !mv pytorch-deep-learning/going_modular .
    !mv pytorch-deep-learning/helper_functions.py .
    !rm -rf pytorch-deep-learning
    from going_modular import data_setup, engine
    from going_modular.utils import download_data, set_seeds
    from helper_functions import plot_loss_curves

In [ ]:
# Print versions
print(f"torch version: {torch.__version__}")
print(f"torchvision version: {torchvision.__version__}")

In [ ]:
# Setup device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## Part 1: Get Data

In [ ]:
# Download 20% pizza, steak, sushi data
data_20_percent_path = download_data(
    source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi_20_percent.zip",
    destination="pizza_steak_sushi_20_percent"
)

data_20_percent_path

In [ ]:
# Setup directory paths
train_dir = data_20_percent_path / "train"
test_dir = data_20_percent_path / "test"

## Part 2: Create EfficientNet-B2 Model

### EfficientNet-B2 Characteristics:
- **Architecture**: CNN with compound scaling
- **Parameters**: ~9M
- **Input size**: 260×260 (but we'll use 224×224)
- **Advantages**: Fast, efficient, good accuracy
- **Ideal for**: Mobile apps, edge devices, real-time applications

In [ ]:
# Quick setup of EfficientNet-B2 (before creating reusable function)

# 1. Get pretrained weights
effnetb2_weights = torchvision.models.EfficientNet_B2_Weights.DEFAULT

# 2. Get transforms from weights
effnetb2_transforms = effnetb2_weights.transforms()

# 3. Create model with pretrained weights
effnetb2 = torchvision.models.efficientnet_b2(weights=effnetb2_weights)

# 4. Freeze base layers (feature extraction)
for param in effnetb2.features.parameters():
    param.requires_grad = False

In [ ]:
# Check classifier head before modification
effnetb2.classifier

In [ ]:
# 5. Update classifier head for 3 classes
effnetb2.classifier = nn.Sequential(
    nn.Dropout(p=0.3, inplace=True),
    nn.Linear(
        in_features=1408,  # EfficientNet-B2 feature dimension
        out_features=3      # pizza, steak, sushi
    )
)

In [ ]:
# View model summary (uncomment for full output)
# summary(
#     model=effnetb2,
#     input_size=(1, 3, 224, 224),
#     col_names=["input_size", "output_size", "num_params", "trainable"],
#     col_width=20,
#     row_settings=["var_names"]
# )

### Create Reusable Model Function

In [ ]:
def create_effnetb2_model(num_classes: int=3, seed: int=42):
    """
    Creates an EfficientNetB2 feature extractor model and transforms.

    Args:
        num_classes (int, optional): Number of output classes. Defaults to 3.
        seed (int, optional): Random seed for reproducibility. Defaults to 42.

    Returns:
        tuple: (model, transforms)
            - model: EfficientNetB2 with custom classifier
            - transforms: Pretrained model's transforms
    """
    # 1. Get pretrained weights and transforms
    weights = torchvision.models.EfficientNet_B2_Weights.DEFAULT
    transforms = weights.transforms()
    
    # 2. Create model
    model = torchvision.models.efficientnet_b2(weights=weights)

    # 3. Freeze feature layers
    for param in model.features.parameters():
        param.requires_grad = False

    # 4. Set seeds for reproducibility
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    
    # 5. Update classifier
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(in_features=1408, out_features=num_classes)
    )
    
    return model, transforms

In [ ]:
# Create EfficientNet-B2 model
effnetb2, effnetb2_transforms = create_effnetb2_model(
    num_classes=3,
    seed=42
)

In [ ]:
# Add data augmentation to training transforms
effnetb2_transforms = torchvision.transforms.Compose([
    torchvision.transforms.TrivialAugmentWide(),  # Strong augmentation
    effnetb2_transforms  # Pretrained model transforms
])

print(effnetb2_transforms)

In [ ]:
# Print model summary (uncomment for full output)
# summary(
#     effnetb2,
#     input_size=(1, 3, 224, 224),
#     col_names=["input_size", "output_size", "num_params", "trainable"],
#     col_width=20,
#     row_settings=["var_names"]
# )

## Part 3: Train EfficientNet-B2

In [ ]:
# Set batch size
BATCH_SIZE = 32

# Create DataLoaders
train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(
    train_dir=train_dir,
    test_dir=test_dir,
    transform=effnetb2_transforms,
    batch_size=BATCH_SIZE
)

print(f"Training batches: {len(train_dataloader)}")
print(f"Testing batches: {len(test_dataloader)}")
print(f"Classes: {class_names}")

In [ ]:
# Setup training
optimizer = torch.optim.Adam(
    params=effnetb2.parameters(),
    lr=1e-3
)

loss_fn = torch.nn.CrossEntropyLoss()

# Train the model
set_seeds()
effnetb2_results = engine.train(
    model=effnetb2,
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    epochs=5,
    device=device
)

In [ ]:
# Plot training curves
plot_loss_curves(effnetb2_results)

## Part 4: Save EfficientNet-B2 Model

In [ ]:
from going_modular import utils

In [ ]:
# Save the model
utils.save_model(
    model=effnetb2,
    target_dir="models",
    model_name="09_pretrained_effnetb2_feature_extractor_pizza_steak_sushi_20_percent.pth"
)

In [ ]:
# Check model size
from pathlib import Path

# Get size in bytes, convert to MB
pretrained_effnetb2_model_size = (
    Path("models/09_pretrained_effnetb2_feature_extractor_pizza_steak_sushi_20_percent.pth").stat().st_size 
    / (1024*1024)
)

print(f"Pretrained EfficientNetB2 model size: {pretrained_effnetb2_model_size:.2f} MB")

## Part 5: Collect EfficientNet-B2 Statistics

For deployment decisions, we need to track:
- **Test accuracy**: How well does it perform?
- **Test loss**: Is it well-calibrated?
- **Total parameters**: Model complexity
- **Model size**: Storage/memory requirements
- **Inference time**: Speed for real-time use

In [ ]:
# Count total parameters
effnetb2_total_params = sum(
    torch.numel(param) for param in effnetb2.parameters()
)

print(f"Total parameters: {effnetb2_total_params:,}")

# Create statistics dictionary
effnetb2_stats = {
    "test_loss": effnetb2_results["test_loss"][-1],  # Final test loss
    "test_acc": effnetb2_results["test_acc"][-1],    # Final test accuracy
    "total_params": effnetb2_total_params,
    "model_size": pretrained_effnetb2_model_size
}

print(f"\nEfficientNet-B2 Stats:")
for key, value in effnetb2_stats.items():
    print(f"  {key}: {value}")

## Part 6: Create Vision Transformer (ViT) Model

### ViT-B/16 Characteristics:
- **Architecture**: Transformer (pure attention, no convolutions)
- **Parameters**: ~86M (much larger than EfficientNet-B2)
- **Input size**: 224×224 split into 16×16 patches
- **Advantages**: State-of-the-art accuracy, global attention
- **Disadvantages**: Slower, needs more compute
- **Ideal for**: Cloud/server deployment, accuracy-critical applications

In [ ]:
def create_vit_model(num_classes: int=3, seed: int=42):
    """
    Creates a ViT-B/16 feature extractor model and transforms.

    Args:
        num_classes (int, optional): Number of target classes. Defaults to 3.
        seed (int, optional): Random seed for reproducibility. Defaults to 42.

    Returns:
        tuple: (model, transforms)
            - model: ViT-B/16 with custom classifier head
            - transforms: Pretrained model's transforms
    """
    # 1. Get pretrained weights
    weights = torchvision.models.ViT_B_16_Weights.DEFAULT
    
    # 2. Get transforms
    transforms = weights.transforms()
    
    # 3. Create model
    model = torchvision.models.vit_b_16(weights=weights)

    # 4. Freeze base layers
    # For ViT, freeze all layers in the encoder
    for param in model.parameters():
        param.requires_grad = False

    # 5. Set seeds
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    
    # 6. Update classifier head
    # ViT has a 'heads' attribute instead of 'classifier'
    model.heads = nn.Sequential(
        nn.Linear(
            in_features=768,  # ViT-B/16 hidden dimension
            out_features=num_classes
        )
    )
    
    return model, transforms

In [ ]:
# Create ViT model and transforms
vit, vit_transforms = create_vit_model(
    num_classes=3,
    seed=42
)

In [ ]:
# Print ViT model summary (uncomment for full output)
# summary(
#     vit,
#     input_size=(1, 3, 224, 224),
#     col_names=["input_size", "output_size", "num_params", "trainable"],
#     col_width=20,
#     row_settings=["var_names"]
# )

## Part 7: Train Vision Transformer

In [ ]:
# Create DataLoaders for ViT
# Note: Using ViT transforms (different from EfficientNet)
BATCH_SIZE = 32

train_dataloader_vit, test_dataloader_vit, class_names = data_setup.create_dataloaders(
    train_dir=train_dir,
    test_dir=test_dir,
    transform=vit_transforms,  # ViT-specific transforms
    batch_size=BATCH_SIZE
)

print(f"Training batches: {len(train_dataloader_vit)}")
print(f"Testing batches: {len(test_dataloader_vit)}")

In [ ]:
# Setup training
optimizer = torch.optim.Adam(
    params=vit.parameters(),
    lr=1e-3
)

loss_fn = torch.nn.CrossEntropyLoss()

print(f"Training on: {device}\n")

# Train ViT model
set_seeds()
vit_results = engine.train(
    model=vit,
    train_dataloader=train_dataloader_vit,
    test_dataloader=test_dataloader_vit,
    optimizer=optimizer,
    loss_fn=loss_fn,
    epochs=5,  # Same as EfficientNet for fair comparison
    device=device
)

In [ ]:
# Plot training curves
plot_loss_curves(vit_results)

In [ ]:
# Save ViT model
utils.save_model(
    model=vit,
    target_dir="models",
    model_name="09_pretrained_vit_feature_extractor_pizza_steak_sushi_20_percent.pth"
)

In [ ]:
# Check ViT model size
from pathlib import Path

pretrained_vit_model_size = (
    Path("models/09_pretrained_vit_feature_extractor_pizza_steak_sushi_20_percent.pth").stat().st_size 
    / (1024*1024)
)

print(f"Pretrained ViT model size: {pretrained_vit_model_size:.2f} MB")

## Part 8: Collect ViT Statistics

In [ ]:
# Count ViT parameters
vit_total_params = sum(
    torch.numel(param) for param in vit.parameters()
)

print(f"Total parameters: {vit_total_params:,}")

# Create ViT statistics dictionary
vit_stats = {
    "test_loss": vit_results["test_loss"][-1],
    "test_acc": vit_results["test_acc"][-1],
    "total_params": vit_total_params,
    "model_size": pretrained_vit_model_size
}

print(f"\nViT Stats:")
for key, value in vit_stats.items():
    print(f"  {key}: {value}")

In [ ]:
# Display both model stats side by side
print("\n" + "="*50)
print("MODEL COMPARISON")
print("="*50)

print("\nViT-B/16:")
for k, v in vit_stats.items():
    print(f"  {k}: {v}")

print("\nEfficientNet-B2:")
for k, v in effnetb2_stats.items():
    print(f"  {k}: {v}")

## Part 9: Measure Prediction Speed

**Critical for deployment**: How fast can the model make predictions?

### Why Speed Matters:
- **Real-time apps**: Need < 100ms response time
- **Batch processing**: Throughput matters
- **Cost**: Slower = more compute = higher costs
- **User experience**: Nobody likes waiting

In [ ]:
import numpy as np

In [ ]:
# Get all test image paths
from pathlib import Path

print(f"[INFO] Finding all .jpg files in: {test_dir}\n")
test_data_paths = list(Path(test_dir).glob("*/*.jpg"))

print(f"Found {len(test_data_paths)} test images")
print(f"First image: {test_data_paths[0]}")

In [ ]:
import pathlib
import torch
from PIL import Image
from timeit import default_timer as timer
from tqdm.auto import tqdm
from typing import List, Dict

In [ ]:
def pred_and_store(
    paths: List[pathlib.Path],
    model: torch.nn.Module,
    transform: torchvision.transforms,
    class_names: List[str],
    device: str = "cpu"
) -> List[Dict]:
    """
    Makes predictions on images and stores results with timing information.

    Args:
        paths: List of image file paths
        model: Trained PyTorch model
        transform: Transform pipeline for preprocessing
        class_names: List of class names
        device: Device to run inference on

    Returns:
        List of dictionaries containing:
            - image_path: Path to image
            - class_name: True class name
            - pred_prob: Prediction probability
            - pred_class: Predicted class name
            - correct: Whether prediction was correct
            - time_of_pred: Prediction time in seconds
    """
    # Create empty list to store prediction dicts
    pred_list = []

    # Put model in eval mode and on target device
    model.to(device)
    model.eval()

    # Loop through images
    with torch.inference_mode():
        for path in tqdm(paths):
            # Create empty dict for storing prediction info
            pred_dict = {}

            # Store image path
            pred_dict["image_path"] = path

            # Get true class name from path
            # Path structure: data/test/class_name/image.jpg
            class_name = path.parent.stem
            pred_dict["class_name"] = class_name

            # Load and transform image
            img = Image.open(path)
            transformed_image = transform(img).unsqueeze(0).to(device)

            # Time the prediction
            start_time = timer()
            
            # Make prediction
            pred_logits = model(transformed_image)
            pred_prob = torch.softmax(pred_logits, dim=1)
            pred_label = torch.argmax(pred_prob, dim=1)
            
            end_time = timer()

            # Store prediction results
            pred_dict["pred_prob"] = pred_prob.max().cpu().item()
            pred_dict["pred_class"] = class_names[pred_label.cpu()]
            
            # Check if prediction correct
            pred_dict["correct"] = class_name == pred_dict["pred_class"]
            
            # Store prediction time
            pred_dict["time_of_pred"] = end_time - start_time

            # Add to list
            pred_list.append(pred_dict)

    return pred_list

### Make Predictions with EfficientNet-B2

In [ ]:
# Make predictions on all test images with EfficientNet-B2
effnetb2_test_pred_dicts = pred_and_store(
    paths=test_data_paths,
    model=effnetb2,
    transform=effnetb2_transforms,
    class_names=class_names,
    device=device
)

In [ ]:
# Inspect first 2 predictions
effnetb2_test_pred_dicts[:2]

In [ ]:
# Convert to DataFrame for easier analysis
import pandas as pd

effnetb2_test_pred_df = pd.DataFrame(effnetb2_test_pred_dicts)
effnetb2_test_pred_df.head()

In [ ]:
# Check prediction accuracy
print("EfficientNet-B2 Predictions:")
print(effnetb2_test_pred_df.correct.value_counts())
print(f"\nAccuracy: {effnetb2_test_pred_df.correct.mean():.2%}")

In [ ]:
# Calculate average prediction time
effnetb2_average_time_per_pred = round(
    effnetb2_test_pred_df.time_of_pred.mean(),
    4
)

print(f"EfficientNet-B2 average time per prediction: {effnetb2_average_time_per_pred} seconds")
print(f"That's {1/effnetb2_average_time_per_pred:.1f} predictions per second")

In [ ]:
# Add prediction time to stats
effnetb2_stats["time_per_pred_cpu"] = effnetb2_average_time_per_pred
effnetb2_stats

### Make Predictions with Vision Transformer

In [ ]:
# Make predictions with ViT
vit_test_pred_dicts = pred_and_store(
    paths=test_data_paths,
    model=vit,
    transform=vit_transforms,
    class_names=class_names,
    device=device
)

In [ ]:
# Convert to DataFrame
vit_test_pred_df = pd.DataFrame(vit_test_pred_dicts)
vit_test_pred_df.head()

In [ ]:
# Check ViT accuracy
print("Vision Transformer Predictions:")
print(vit_test_pred_df.correct.value_counts())
print(f"\nAccuracy: {vit_test_pred_df.correct.mean():.2%}")

In [ ]:
# Calculate average ViT prediction time
vit_average_time_per_pred = round(
    vit_test_pred_df.time_of_pred.mean(),
    4
)

print(f"ViT average time per prediction: {vit_average_time_per_pred} seconds")
print(f"That's {1/vit_average_time_per_pred:.1f} predictions per second")

In [ ]:
# Add to ViT stats
vit_stats["time_per_pred_cpu"] = vit_average_time_per_pred
vit_stats

## Part 10: Compare Models for Deployment

Now we have all the metrics to make an informed deployment decision!

In [ ]:
# Create comparison DataFrame
comparison_df = pd.DataFrame([
    effnetb2_stats,
    vit_stats
], index=["EfficientNet-B2", "ViT-B/16"])

print("\n" + "="*70)
print("MODEL DEPLOYMENT COMPARISON")
print("="*70)
print(comparison_df)

# Calculate speedup
speedup = vit_stats["time_per_pred_cpu"] / effnetb2_stats["time_per_pred_cpu"]
print(f"\nEfficientNet-B2 is {speedup:.1f}x faster than ViT")

## Summary: Deployment Decision Matrix

### Model Comparison Results:

| Metric | EfficientNet-B2 | ViT-B/16 | Winner |
|--------|-----------------|----------|--------|
| **Accuracy** | ~75-85% | ~75-90% | ViT (slight) |
| **Speed** | ~0.05s/image | ~0.2s/image | **EfficientNet (4x faster)** |
| **Model Size** | ~35 MB | ~330 MB | **EfficientNet (10x smaller)** |
| **Parameters** | ~9M | ~86M | **EfficientNet (10x fewer)** |

### Deployment Recommendations:

**Use EfficientNet-B2 when:**
- ✅ Deploying to mobile devices
- ✅ Need real-time predictions (< 100ms)
- ✅ Limited compute budget
- ✅ Edge deployment (IoT, embedded)
- ✅ Batch processing large datasets

**Use ViT-B/16 when:**
- ✅ Cloud/server deployment
- ✅ Accuracy is paramount
- ✅ Latency not critical (batch processing)
- ✅ GPU always available
- ✅ Research/experimentation

### Typical Deployment Scenarios:

**1. Mobile Food App (e.g., Calorie Counter)**
- **Choose**: EfficientNet-B2
- **Why**: Fast, small, works on-device
- **Deployment**: Mobile model (CoreML, TFLite)

**2. Restaurant POS System**
- **Choose**: EfficientNet-B2
- **Why**: Real-time, reliable, efficient
- **Deployment**: Edge server or tablet

**3. Food Discovery Service (Cloud)**
- **Choose**: ViT-B/16
- **Why**: Best accuracy, batch processing OK
- **Deployment**: Cloud API (AWS, GCP, Azure)

**4. Social Media Auto-Tagging**
- **Choose**: EfficientNet-B2
- **Why**: Process millions of images/day
- **Deployment**: Distributed cloud processing

### Further Optimizations:

**For EfficientNet-B2:**
- Model quantization (INT8): 4x smaller, 2x faster
- Pruning: Remove redundant weights
- Knowledge distillation: Train smaller student model

**For ViT:**
- Use smaller ViT-S or ViT-Ti variants
- Reduce image resolution
- Batch processing on GPU

### Production Checklist:

- [ ] Model accuracy acceptable (> 70%)
- [ ] Inference time acceptable (< target latency)
- [ ] Model size fits deployment constraints
- [ ] Tested on edge cases
- [ ] A/B testing plan ready
- [ ] Monitoring/logging set up
- [ ] Fallback strategy defined
- [ ] Continuous improvement pipeline

### Next Steps:

1. **Export model** (ONNX, TorchScript, etc.)
2. **Create API** (FastAPI, Flask)
3. **Containerize** (Docker)
4. **Deploy** (AWS Lambda, GCP Cloud Run, etc.)
5. **Monitor** (Latency, accuracy, errors)
6. **Iterate** (Collect edge cases, retrain)